In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)

In [ ]:
# Configurazione stile grafici
sns.set_theme(style="whitegrid")

# Trova il file CSV nella cartella scaricata da kagglehub
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
full_path = os.path.join(path, csv_file)

# Carica il dataset
df = pd.read_csv(full_path)

print(f"Shape del Dataset: {df.shape}")
print("-" * 50)
df.head()

In [ ]:
# 1. Rimuoviamo la colonna ID
df_clean = df.drop(columns=['customerID']).copy()

# 2. Pulizia TotalCharges: convertiamo gli spazi vuoti " " in NaN e poi a float
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'].str.strip(), errors='coerce')

# I valori NaN (nuovi clienti con tenure=0) li impostiamo a 0
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(0)

# 3. Mappiamo la variabile Target: 'No' -> 0, 'Yes' -> 1
df_clean['Churn'] = df_clean['Churn'].map({'No': 0, 'Yes': 1})

# Verifica pulizia
print(f"Valori nulli rimasti in TotalCharges: {df_clean['TotalCharges'].isnull().sum()}")
print(f"Tipo di dato di TotalCharges: {df_clean['TotalCharges'].dtype}")
print(f"Tipo di dato di Churn: {df_clean['Churn'].dtype}")

In [ ]:
# COLONNE CON CATEGORIA
# Ora tocca alle colonne delle categorie che vanno modificate in numeri e moltiplicate
# Se una categoria Contratto può essere "Month-to-month", "One year", "Two year"
# da una colonna, ne creo 3 e gli assegno un valore binario (si/no)

# Identifichiamo le colonne categoriche (tipo 'object') tranne Churn che è già 0/1
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

# Applichiamo il One-Hot Encoding su tutte le variabili categoriche
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True, dtype=int)

print(f"Shape originale: {df_clean.shape}")
print(f"Shape dopo One-Hot Encoding: {df_encoded.shape}")
print("-" * 50)
print("Nuove colonne create (prime 5):")
print(df_encoded.columns[3:8].tolist())

In [ ]:

# 1. Separiamo Feature (X) e Target (y)
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

# 2. Split Train-Test 80-20 come al solito
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dimensioni X_train: {X_train.shape}")
print(f"Dimensioni X_test:  {X_test.shape}")
print(f"\nProporzione Churn nel Train Set:\n{y_train.value_counts(normalize=True)}")

In [ ]:
# 1. Inizializziamo XGBoost per la classificazione
xgb_model = xgb.XGBClassifier(
    n_estimators=100,        # Numero di alberi
    learning_rate=0.05,      # Passo di apprendimento
    max_depth=4,             # Profondità massima di ogni albero
    random_state=42,
    eval_metric='logloss'
)

# 2. Addestramento sui dati NON SCALATI (X_train)
xgb_model.fit(X_train, y_train)

# 3. Predizione sul Test Set
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1] # converte i risultati in percentuali

# 4. Valutazione
acc_xgb = accuracy_score(y_test, y_pred_xgb) # calcola la percentuale di risposte esatte
auc_xgb = roc_auc_score(y_test, y_proba_xgb) # controlla le probabilità continuative

print(f"XGBoost Test Accuracy: {acc_xgb:.4f}")
print(f"XGBoost Test ROC-AUC:  {auc_xgb:.4f}")

XGBoost Test Accuracy: 0.7999
XGBoost Test ROC-AUC:  0.8465


In [ ]:
# Estraiamo le 10 feature più importanti per XGBoost
feature_importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
top_10_features = feature_importances.nlargest(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_10_features.values, y=top_10_features.index, palette="viridis")
plt.title("Top 10 Feature Più Importanti per XGBoost (Telco Churn)")
plt.xlabel("Importanza Relativa")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()